# Risk-aware robust fine-tuning of the full-Haar compiler

This notebook starts from the frozen nominal neural pulse compiler and fine-tunes the same architecture under the routine uncertainty model used in the manuscript.

The **training RF-gain envelope is fixed at ±5%**. Each training microbatch contains eight random joint uncertainty scenarios plus the two exact RF boundary cases. The loss is

\[
\mathcal L=
\mathcal L_{\mathrm{nominal}}
+\mathcal L_{\mathrm{mean}}
+1.5\,\mathcal L_{\mathrm{CVaR}}
+\mathcal L_{\mathrm{control}},
\]

where CVaR is computed separately for each target gate across its uncertainty scenarios. With ten scenarios and \(\alpha=0.20\), the CVaR term averages the two largest infidelities for each gate.

The non-RF training envelope is held fixed at:

- common carrier/B0 offset: ±2 Hz,
- independent spin-frequency offsets: ±2 Hz per spin,
- independent J-coupling offsets: ±0.25 Hz per coupling,
- static RF phase-reference error: ±1°,
- clock-scale error: ±2 ppm,
- slice timing jitter: 10 ps RMS.

After training, the selected model is evaluated at RF envelopes of ±5%, ±7.5%, ±10%, and a conservative ±15% stress envelope. Only the ±5% envelope is used during optimization.

In [ ]:
# Imports and runtime setup
import copy
import hashlib
import json
import math
import platform
import random
import shutil
import subprocess
import sys
from dataclasses import dataclass, fields
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA-capable GPU is required for the full robust-training run.")

DEVICE = torch.device("cuda")
torch.backends.cuda.matmul.allow_tf32 = True

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def sha256_file(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(block_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

In [ ]:
# Load the frozen nominal checkpoint produced by the nominal-training workflow.
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

NOMINAL_CHECKPOINT = REPO_ROOT / "data" / "checkpoints" / "nominal_full_checkpoint.pt"
if not NOMINAL_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Nominal checkpoint not found: {NOMINAL_CHECKPOINT}\n"
        "Place the frozen nominal full checkpoint in data/checkpoints/ before running this notebook."
    )

nominal_checkpoint = torch.load(
    NOMINAL_CHECKPOINT,
    map_location=DEVICE,
    weights_only=False,
)

if "model" not in nominal_checkpoint or "config" not in nominal_checkpoint:
    raise ValueError("The nominal checkpoint must contain both 'model' and 'config' entries.")

CONFIG_DICT = nominal_checkpoint["config"]

print("Nominal checkpoint:", NOMINAL_CHECKPOINT)
print("SHA-256:", sha256_file(NOMINAL_CHECKPOINT))
print("Nominal checkpoint step:", nominal_checkpoint.get("step"))
print("Nominal validation statistics:", nominal_checkpoint.get("val_stats"))

In [ ]:
# Exact network architecture and three-spin XX+YY+ZZ model.
@dataclass
class Config:
    v_hz: tuple = (-921.0, 40.75, 700.0)
    j_hz: tuple = (-64.0, 24.4, 34.1)
    n_slices: int = 300
    dt_s: float = 35e-6
    rabi_max_hz: float = 50_000.0

    width: int = 256
    hidden_layers: int = 10
    dropout: float = 0.25

    batch_size: int = 32
    train_steps: int = 50_000
    lr: float = 5e-4
    weight_decay: float = 1e-3
    grad_clip: float = 1.0
    smooth_lambda: float = 1e-4
    power_lambda: float = 1e-6

    val_gates: int = 512
    val_batch: int = 64
    validate_every: int = 100
    print_every: int = 20

    seed: int = 20260917
    out_dir: str = "outputs/nominal"

allowed_fields = {item.name for item in fields(Config)}
CFG = Config(**{key: value for key, value in CONFIG_DICT.items() if key in allowed_fields})
CFG.v_hz = tuple(CFG.v_hz)
CFG.j_hz = tuple(CFG.j_hz)

assert CFG.n_slices == 300
assert abs(CFG.dt_s - 35e-6) < 1e-12
assert CFG.width == 256
assert CFG.hidden_layers == 10
assert abs(CFG.dropout - 0.25) < 1e-12

print(CFG)

def kron3(a, b, c):
    return torch.kron(torch.kron(a, b), c)

class NMRSystem:
    def __init__(self, cfg, device):
        self.cfg = cfg
        self.device = device
        complex_type = torch.complex64

        I = torch.eye(2, dtype=complex_type, device=device)
        X = torch.tensor([[0, 1], [1, 0]], dtype=complex_type, device=device)
        Y = torch.tensor([[0, -1j], [1j, 0]], dtype=complex_type, device=device)
        Z = torch.tensor([[1, 0], [0, -1]], dtype=complex_type, device=device)

        self.I2 = I
        self.I4 = torch.eye(4, dtype=complex_type, device=device)
        self.I8 = torch.eye(8, dtype=complex_type, device=device)
        self.X, self.Y, self.Z = X, Y, Z

        self.Xs = [kron3(X, I, I), kron3(I, X, I), kron3(I, I, X)]
        self.Ys = [kron3(Y, I, I), kron3(I, Y, I), kron3(I, I, Y)]
        self.Zs = [kron3(Z, I, I), kron3(I, Z, I), kron3(I, I, Z)]

        self.Hx = sum(self.Xs)
        self.Hy = sum(self.Ys)

        self.pair_ops = []
        for i, j in [(0, 1), (0, 2), (1, 2)]:
            self.pair_ops.append(
                self.Xs[i] @ self.Xs[j]
                + self.Ys[i] @ self.Ys[j]
                + self.Zs[i] @ self.Zs[j]
            )

        v = torch.tensor(cfg.v_hz, dtype=torch.float32, device=device)[None, :]
        j = torch.tensor(cfg.j_hz, dtype=torch.float32, device=device)[None, :]
        self.H0 = self.build_h0(v, j)[0]

    def build_h0(self, v_hz, j_hz):
        H = torch.zeros((v_hz.shape[0], 8, 8), dtype=torch.complex64, device=self.device)
        for i in range(3):
            H = H + math.pi * v_hz[:, i, None, None].to(H.dtype) * self.Zs[i]
        for k in range(3):
            H = H + math.pi * j_hz[:, k, None, None].to(H.dtype) * self.pair_ops[k]
        return H

SYS = NMRSystem(CFG, DEVICE)

def sample_haar_quaternions(n, device, generator=None):
    q = torch.randn(n, 4, device=device, generator=generator)
    return q / q.norm(dim=-1, keepdim=True).clamp_min(1e-12)

def quaternion_to_u2(q, system):
    q0, qx, qy, qz = q.unbind(-1)
    I, X, Y, Z = system.I2, system.X, system.Y, system.Z
    return (
        q0[:, None, None].to(I.dtype) * I
        - 1j * (
            qx[:, None, None].to(I.dtype) * X
            + qy[:, None, None].to(I.dtype) * Y
            + qz[:, None, None].to(I.dtype) * Z
        )
    )

def lift_target(U2, system):
    return torch.einsum("bij,kl->bikjl", U2, system.I4).reshape(-1, 8, 8)

class PulseCompiler(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        layers = []
        input_dim = 4
        for _ in range(cfg.hidden_layers):
            layers.extend([nn.Linear(input_dim, cfg.width), nn.GELU()])
            if cfg.dropout > 0:
                layers.append(nn.Dropout(cfg.dropout))
            input_dim = cfg.width

        self.backbone = nn.Sequential(*layers)
        self.head = nn.Linear(input_dim, 2 * cfg.n_slices)
        self.cfg = cfg

    def forward(self, q):
        raw = self.head(self.backbone(q)).view(-1, self.cfg.n_slices, 2)
        radius_sq = (raw * raw).sum(dim=-1, keepdim=True)
        return self.cfg.rabi_max_hz * raw / torch.sqrt(1.0 + radius_sq)

def unitary_fidelity(target, actual):
    overlap = (target.conj() * actual).sum(dim=(-2, -1))
    return overlap.abs().square() / 64.0

def control_regularizer(controls):
    difference = controls[:, 1:] - controls[:, :-1]
    smoothness = (difference.square().sum(-1) / (CFG.rabi_max_hz ** 2)).mean()
    power = (controls.square().sum(-1) / (CFG.rabi_max_hz ** 2)).mean()
    return smoothness, power

def propagate_nominal(controls, system):
    ux, uy = controls[..., 0], controls[..., 1]
    H = (
        system.H0[None, None, :, :]
        + math.pi * ux[..., None, None].to(torch.complex64) * system.Hx
        + math.pi * uy[..., None, None].to(torch.complex64) * system.Hy
    )
    U_steps = torch.matrix_exp((-1j * CFG.dt_s) * H)
    U = system.I8[None, :, :].expand(len(controls), -1, -1).clone()
    for t in range(CFG.n_slices):
        U = U_steps[:, t] @ U
    return U

def summarize(values):
    values = np.asarray(values, dtype=float)
    return {
        "mean": float(values.mean()),
        "std": float(values.std()),
        "median": float(np.median(values)),
        "p05": float(np.quantile(values, 0.05)),
        "p01": float(np.quantile(values, 0.01)),
        "min": float(values.min()),
        "max": float(values.max()),
    }

In [ ]:
# Load the frozen nominal model and initialize the trainable copy from the same weights.
NOMINAL_MODEL = PulseCompiler(CFG).to(DEVICE)
NOMINAL_MODEL.load_state_dict(nominal_checkpoint["model"])
NOMINAL_MODEL.eval()
for parameter in NOMINAL_MODEL.parameters():
    parameter.requires_grad_(False)

MODEL = PulseCompiler(CFG).to(DEVICE)
MODEL.load_state_dict(nominal_checkpoint["model"])
MODEL.train()

print("Parameters:", sum(parameter.numel() for parameter in MODEL.parameters()))
print("The robust model starts from the frozen nominal weights.")

In [ ]:
# Robust-training configuration.
#
# The archived experiment used a 5,800-step cosine horizon. Only the first
# 1,800 optimization steps were executed for the reported ±5% training run.
# Keeping the original horizon is necessary to reproduce the learning-rate
# trajectory of the selected checkpoint exactly.
ROBUST_SEED = 20260918
set_seed(ROBUST_SEED)

MICRO_GATES = 4
GRAD_ACCUM_STEPS = 4
RANDOM_SCENARIOS = 8
CVAR_ALPHA = 0.20

LAMBDA_NOMINAL = 1.0
LAMBDA_MEAN = 1.0
LAMBDA_CVAR = 1.5
ROBUST_SMOOTH_LAMBDA = CFG.smooth_lambda
ROBUST_POWER_LAMBDA = CFG.power_lambda

ROBUST_LR = 1.5e-4
GRAD_CLIP = 1.0

VALIDATE_EVERY = 100
VAL_GATES = 96
VAL_RANDOM_SCENARIOS = 24

TRAIN_RF_ABS = 0.05
MAX_STEPS = 1800
SCHEDULER_HORIZON_STEPS = 5800

ROUTINE = {
    "common_B0_abs_hz": 2.0,
    "independent_v_abs_hz": 2.0,
    "independent_J_abs_hz": 0.25,
    "phase_bias_abs_deg": 1.0,
    "clock_abs_ppm": 2.0,
    "timing_jitter_rms_ps": 10.0,
}

CONSERVATIVE = {
    "common_B0_abs_hz": 6.0,
    "independent_v_abs_hz": 6.0,
    "independent_J_abs_hz": 1.0,
    "rf_gain_abs_fraction": 0.15,
    "phase_bias_abs_deg": 3.0,
    "clock_abs_ppm": 5.0,
    "timing_jitter_rms_ps": 50.0,
}

print(f"Training RF range: ±{100 * TRAIN_RF_ABS:.1f}%")
print("Optimizer steps:", MAX_STEPS)
print("Cosine scheduler horizon:", SCHEDULER_HORIZON_STEPS)

In [ ]:
# Scenario generation and differentiable propagation under joint uncertainty.

def random_training_scenarios(S,rf_abs):
    u=lambda *shape:2*torch.rand(*shape,device=DEVICE)-1
    return {
      'common_B0_hz':u(S)*ROUTINE['common_B0_abs_hz'],
      'v_offset_hz':u(S,3)*ROUTINE['independent_v_abs_hz'],
      'J_offset_hz':u(S,3)*ROUTINE['independent_J_abs_hz'],
      'rf_gain':1+u(S)*rf_abs,
      'phase_bias_rad':torch.deg2rad(u(S)*ROUTINE['phase_bias_abs_deg']),
      'clock_ppm':u(S)*ROUTINE['clock_abs_ppm'],
      'timing_jitter_s':torch.randn(S,CFG.n_slices,device=DEVICE)*(ROUTINE['timing_jitter_rms_ps']*1e-12),
    }

def boundary_rf_scenarios(rf_abs):
    z=torch.zeros; S=2
    return {'common_B0_hz':z(S,device=DEVICE),'v_offset_hz':z(S,3,device=DEVICE),'J_offset_hz':z(S,3,device=DEVICE),'rf_gain':torch.tensor([1-rf_abs,1+rf_abs],device=DEVICE),'phase_bias_rad':z(S,device=DEVICE),'clock_ppm':z(S,device=DEVICE),'timing_jitter_s':z(S,CFG.n_slices,device=DEVICE)}

def cat_scenarios(a,b): return {k:torch.cat([a[k],b[k]],0) for k in a}

def fixed_sobol_scenarios(S, rf_abs, seed, profile="routine"):
    profile_config = (
        {**ROUTINE, "rf_gain_abs_fraction": rf_abs}
        if profile == "routine"
        else CONSERVATIVE
    )
    e=torch.quasirandom.SobolEngine(dimension=10,scramble=True,seed=seed); z=2*e.draw(S).to(DEVICE)-1; k=0
    s={}
    s['common_B0_hz']=z[:,k]*profile_config['common_B0_abs_hz']; k+=1
    s['v_offset_hz']=z[:,k:k+3]*profile_config['independent_v_abs_hz']; k+=3
    s['J_offset_hz']=z[:,k:k+3]*profile_config['independent_J_abs_hz']; k+=3
    s['rf_gain']=1+z[:,k]*profile_config['rf_gain_abs_fraction']; k+=1
    s['phase_bias_rad']=torch.deg2rad(z[:,k]*profile_config['phase_bias_abs_deg']); k+=1
    s['clock_ppm']=z[:,k]*profile_config['clock_abs_ppm']; k+=1
    gen=torch.Generator(device=DEVICE); gen.manual_seed(seed+991)
    s['timing_jitter_s']=torch.randn(S,CFG.n_slices,device=DEVICE,generator=gen)*(profile_config['timing_jitter_rms_ps']*1e-12)
    return s

def propagate_scenarios(controls,s,sysobj):
    G,T,_=controls.shape; S=len(s['rf_gain'])
    v0=torch.tensor(CFG.v_hz,dtype=torch.float32,device=DEVICE); j0=torch.tensor(CFG.j_hz,dtype=torch.float32,device=DEVICE)
    v=v0[None,:]+s['common_B0_hz'][:,None]+s['v_offset_hz']; j=j0[None,:]+s['J_offset_hz']; H0s=sysobj.build_h0(v,j)
    c=controls[:,None,:,:].expand(G,S,-1,-1); gain=s['rf_gain'][None,:,None]; ph=s['phase_bias_rad'][None,:,None]; cp,sp=torch.cos(ph),torch.sin(ph)
    ux=(gain*(cp*c[...,0]-sp*c[...,1])).reshape(G*S,T); uy=(gain*(sp*c[...,0]+cp*c[...,1])).reshape(G*S,T)
    H0=H0s[None,:,:,:].expand(G,S,-1,-1).reshape(G*S,8,8)
    H=H0[:,None,:,:]+math.pi*ux[...,None,None].to(torch.complex64)*sysobj.Hx+math.pi*uy[...,None,None].to(torch.complex64)*sysobj.Hy
    dt=CFG.dt_s*(1+s['clock_ppm']*1e-6)[:,None]+s['timing_jitter_s']; dt=torch.clamp(dt,min=CFG.dt_s*.5); dt=dt[None,:,:].expand(G,S,-1).reshape(G*S,T)
    Usteps=torch.matrix_exp((-1j*dt[...,None,None])*H); U=sysobj.I8[None,:,:].expand(G*S,-1,-1).clone()
    for t in range(T): U=Usteps[:,t]@U
    return U.reshape(G,S,8,8)

def scenario_fidelities(targets,U):
    G,S=U.shape[:2]; targ=targets[:,None,:,:].expand(G,S,-1,-1); ov=(targ.conj()*U).sum(dim=(-2,-1)); return ov.abs().square()/64.0

def cvar_loss(infid,alpha=CVAR_ALPHA):
    k=max(1,int(math.ceil(alpha*infid.shape[1]))); return torch.topk(infid,k,dim=1,largest=True).values.mean()

def robust_micro_loss(model,q,rf_abs):
    targets=lift_target(quaternion_to_u2(q,SYS),SYS); controls=model(q)
    F0=unitary_fidelity(targets,propagate_nominal(controls,SYS)); L0=(1-F0).mean()
    scenarios=cat_scenarios(random_training_scenarios(RANDOM_SCENARIOS,rf_abs),boundary_rf_scenarios(rf_abs))
    Fs=scenario_fidelities(targets,propagate_scenarios(controls,scenarios,SYS)); infid=1-Fs
    Lm=infid.mean(); Lc=cvar_loss(infid); smooth,power=control_regularizer(controls)
    total=LAMBDA_NOMINAL*L0+LAMBDA_MEAN*Lm+LAMBDA_CVAR*Lc+ROBUST_SMOOTH_LAMBDA*smooth+ROBUST_POWER_LAMBDA*power
    return total,{'loss':float(total.detach()),'Fnom':float(F0.mean().detach()),'Frob':float(Fs.mean().detach()),'cvar_loss':float(Lc.detach())}

In [ ]:
# Fixed validation set and run output.
OUT = REPO_ROOT / "outputs" / "robust_training"
if OUT.exists():
    shutil.rmtree(OUT)

CHECKPOINT_DIR = OUT / "checkpoints"
RAW_DIR = OUT / "raw"
TABLE_DIR = OUT / "tables"
FIGURE_DIR = OUT / "figures"
for directory in [OUT, CHECKPOINT_DIR, RAW_DIR, TABLE_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

RUN_CONFIG = {
    "nominal_checkpoint": str(NOMINAL_CHECKPOINT.relative_to(REPO_ROOT)),
    "nominal_checkpoint_sha256": sha256_file(NOMINAL_CHECKPOINT),
    "seed": ROBUST_SEED,
    "micro_gates": MICRO_GATES,
    "grad_accum_steps": GRAD_ACCUM_STEPS,
    "random_scenarios": RANDOM_SCENARIOS,
    "boundary_scenarios": 2,
    "cvar_alpha": CVAR_ALPHA,
    "weights": {
        "nominal": LAMBDA_NOMINAL,
        "mean": LAMBDA_MEAN,
        "cvar": LAMBDA_CVAR,
    },
    "training_rf_abs_fraction": TRAIN_RF_ABS,
    "max_steps": MAX_STEPS,
    "scheduler_horizon_steps": SCHEDULER_HORIZON_STEPS,
    "routine_uncertainty": ROUTINE,
    "conservative_evaluation": CONSERVATIVE,
}

with open(OUT / "robust_training_config.json", "w") as handle:
    json.dump(RUN_CONFIG, handle, indent=2)

validation_generator = torch.Generator(device=DEVICE)
validation_generator.manual_seed(ROBUST_SEED + 12345)
Q_VAL = sample_haar_quaternions(VAL_GATES, DEVICE, validation_generator)
TARGET_VAL = lift_target(quaternion_to_u2(Q_VAL, SYS), SYS)

np.save(RAW_DIR / "validation_quaternions.npy", Q_VAL.cpu().numpy())

# This seed matches the fixed ±5% validation scenarios used in the archived run.
VAL_SCENARIOS = cat_scenarios(
    fixed_sobol_scenarios(
        VAL_RANDOM_SCENARIOS,
        TRAIN_RF_ABS,
        ROBUST_SEED + 500,
        "routine",
    ),
    boundary_rf_scenarios(TRAIN_RF_ABS),
)

np.savez(
    RAW_DIR / "validation_scenarios.npz",
    **{key: value.cpu().numpy() for key, value in VAL_SCENARIOS.items()},
)

@torch.inference_mode()
def eval_nominal(model, q, targets, batch=16):
    model.eval()
    values = []
    for start in range(0, len(q), batch):
        controls = model(q[start : start + batch])
        values.append(
            unitary_fidelity(
                targets[start : start + batch],
                propagate_nominal(controls, SYS),
            ).cpu()
        )
    return torch.cat(values).numpy()

@torch.inference_mode()
def eval_scenarios(model, q, targets, scenarios, gate_chunk=4, scenario_chunk=6):
    model.eval()

    control_batches = []
    for start in range(0, len(q), 32):
        control_batches.append(model(q[start : start + 32]))
    controls = torch.cat(control_batches)

    n_gates = len(q)
    n_scenarios = len(scenarios["rf_gain"])
    output = np.empty((n_gates, n_scenarios), np.float32)

    for gate_start in range(0, n_gates, gate_chunk):
        gate_end = min(n_gates, gate_start + gate_chunk)

        for scenario_start in range(0, n_scenarios, scenario_chunk):
            scenario_end = min(n_scenarios, scenario_start + scenario_chunk)
            subset = {
                key: value[scenario_start:scenario_end]
                for key, value in scenarios.items()
            }

            fidelity = scenario_fidelities(
                targets[gate_start:gate_end],
                propagate_scenarios(controls[gate_start:gate_end], subset, SYS),
            )
            output[
                gate_start:gate_end,
                scenario_start:scenario_end,
            ] = fidelity.cpu().numpy()

        torch.cuda.empty_cache()

    return output

print(
    "Frozen nominal validation:",
    summarize(eval_nominal(NOMINAL_MODEL, Q_VAL, TARGET_VAL)),
)

In [ ]:
# Risk-aware fine-tuning under the fixed ±5% RF training envelope.
optimizer = torch.optim.AdamW(
    MODEL.parameters(),
    lr=ROBUST_LR,
    weight_decay=CFG.weight_decay,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=SCHEDULER_HORIZON_STEPS,
    eta_min=ROBUST_LR * 0.08,
)

history = []
best_score = -np.inf
best_record = None

def save_checkpoint(path, step, validation):
    torch.save(
        {
            "model": MODEL.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "global_step": step,
            "validation": validation,
            "run_config": RUN_CONFIG,
        },
        path,
    )

for step in range(1, MAX_STEPS + 1):
    MODEL.train()
    optimizer.zero_grad(set_to_none=True)
    microbatch_stats = []

    for _ in range(GRAD_ACCUM_STEPS):
        q = sample_haar_quaternions(MICRO_GATES, DEVICE)
        loss, stats = robust_micro_loss(MODEL, q, TRAIN_RF_ABS)
        (loss / GRAD_ACCUM_STEPS).backward()
        microbatch_stats.append(stats)

    grad_norm = torch.nn.utils.clip_grad_norm_(MODEL.parameters(), GRAD_CLIP)
    optimizer.step()
    scheduler.step()

    if step == 1 or step % 20 == 0:
        print(
            f"step {step:4d}/{MAX_STEPS} "
            f"loss={np.mean([x['loss'] for x in microbatch_stats]):.5f} "
            f"Fnom={np.mean([x['Fnom'] for x in microbatch_stats]):.5f} "
            f"Frob={np.mean([x['Frob'] for x in microbatch_stats]):.5f} "
            f"cvar={np.mean([x['cvar_loss'] for x in microbatch_stats]):.5f}"
        )

    if step == 1 or step % VALIDATE_EVERY == 0:
        nominal_values = eval_nominal(MODEL, Q_VAL, TARGET_VAL)
        scenario_values = eval_scenarios(
            MODEL,
            Q_VAL,
            TARGET_VAL,
            VAL_SCENARIOS,
        )

        gate_mean = scenario_values.mean(axis=1)
        gate_worst = scenario_values.min(axis=1)
        infidelity = 1.0 - scenario_values

        tail_count = max(
            1,
            int(math.ceil(CVAR_ALPHA * scenario_values.shape[1])),
        )
        gate_cvar = np.partition(
            infidelity,
            infidelity.shape[1] - tail_count,
            axis=1,
        )[:, infidelity.shape[1] - tail_count :].mean(axis=1)

        record = {
            "global_step": step,
            "lr": optimizer.param_groups[0]["lr"],
            "val_nominal_mean": float(nominal_values.mean()),
            "val_nominal_p05": float(np.quantile(nominal_values, 0.05)),
            "val_robust_mean": float(scenario_values.mean()),
            "val_gate_mean_p05": float(np.quantile(gate_mean, 0.05)),
            "val_gate_worst_mean": float(gate_worst.mean()),
            "val_cvar_infidelity_mean": float(gate_cvar.mean()),
            "grad_norm": float(grad_norm),
        }
        history.append(record)

        print(
            f"  VAL nominal={record['val_nominal_mean']:.6f} "
            f"p05={record['val_nominal_p05']:.6f} "
            f"joint={record['val_robust_mean']:.6f} "
            f"worstMean={record['val_gate_worst_mean']:.6f} "
            f"CVaRinf={record['val_cvar_infidelity_mean']:.6f}"
        )

        # This is the checkpoint-selection score used in the archived run.
        score = (
            record["val_robust_mean"]
            + 0.25 * record["val_nominal_mean"]
            - 8.0 * max(0.0, 0.99 - record["val_nominal_mean"])
        )

        if score > best_score:
            best_score = score
            best_record = copy.deepcopy(record)
            save_checkpoint(
                CHECKPOINT_DIR / "robust_best.pt",
                step,
                record,
            )
            print("  saved best checkpoint")

        pd.DataFrame(history).to_csv(
            TABLE_DIR / "robust_training_history.csv",
            index=False,
        )

best_checkpoint = torch.load(
    CHECKPOINT_DIR / "robust_best.pt",
    map_location=DEVICE,
    weights_only=False,
)

MODEL.load_state_dict(best_checkpoint["model"])
optimizer.load_state_dict(best_checkpoint["optimizer"])
scheduler.load_state_dict(best_checkpoint["scheduler"])
MODEL.eval()

torch.save(
    MODEL.state_dict(),
    CHECKPOINT_DIR / "robust_final_state_dict.pt",
)

print("Selected robust checkpoint:")
print(json.dumps(best_record, indent=2))

In [ ]:
# Final held-out comparison: frozen nominal model versus the selected robust model.
FINAL_GATES = 256
FINAL_SCENARIOS = 96
FINAL_SEED = ROBUST_SEED + 777777
RF_SWEEP_GATES = 128

final_generator = torch.Generator(device=DEVICE)
final_generator.manual_seed(FINAL_SEED)

Q_FINAL = sample_haar_quaternions(FINAL_GATES, DEVICE, final_generator)
TARGET_FINAL = lift_target(quaternion_to_u2(Q_FINAL, SYS), SYS)
np.save(RAW_DIR / "final_haar_quaternions.npy", Q_FINAL.cpu().numpy())

EVALUATION_SUITES = {
    "routine_rf05": fixed_sobol_scenarios(
        FINAL_SCENARIOS, 0.05, FINAL_SEED + 1, "routine"
    ),
    "routine_rf075": fixed_sobol_scenarios(
        FINAL_SCENARIOS, 0.075, FINAL_SEED + 2, "routine"
    ),
    "routine_rf10": fixed_sobol_scenarios(
        FINAL_SCENARIOS, 0.10, FINAL_SEED + 3, "routine"
    ),
    "conservative_rf15": fixed_sobol_scenarios(
        FINAL_SCENARIOS, 0.15, FINAL_SEED + 4, "conservative"
    ),
}

for suite_name, scenarios in EVALUATION_SUITES.items():
    np.savez(
        RAW_DIR / f"final_scenarios_{suite_name}.npz",
        **{key: value.cpu().numpy() for key, value in scenarios.items()},
    )

models = {
    "nominal": NOMINAL_MODEL,
    "robust": MODEL,
}

summary_rows = []

for model_name, model in models.items():
    nominal_values = eval_nominal(model, Q_FINAL, TARGET_FINAL)
    np.save(RAW_DIR / f"final_{model_name}_nominal.npy", nominal_values)
    summary_rows.append(
        {
            "model": model_name,
            "suite": "nominal",
            **summarize(nominal_values),
        }
    )

for suite_name, scenarios in EVALUATION_SUITES.items():
    for model_name, model in models.items():
        fidelity = eval_scenarios(
            model,
            Q_FINAL,
            TARGET_FINAL,
            scenarios,
        )
        np.save(
            RAW_DIR / f"final_{model_name}_{suite_name}.npy",
            fidelity,
        )

        infidelity = 1.0 - fidelity
        tail_count = max(1, int(math.ceil(0.10 * fidelity.shape[1])))
        cvar10 = np.partition(
            infidelity,
            infidelity.shape[1] - tail_count,
            axis=1,
        )[:, infidelity.shape[1] - tail_count :].mean(axis=1)

        summary_rows.append(
            {
                "model": model_name,
                "suite": suite_name,
                **summarize(fidelity.ravel()),
                "gate_mean_mean": float(fidelity.mean(axis=1).mean()),
                "gate_worst_mean": float(fidelity.min(axis=1).mean()),
                "mean_gate_cvar10_infidelity": float(cvar10.mean()),
            }
        )

        pd.DataFrame(
            {
                "gate_index": np.arange(fidelity.shape[0]),
                "mean_fidelity": fidelity.mean(axis=1),
                "worst_fidelity": fidelity.min(axis=1),
                "cvar10_infidelity": cvar10,
            }
        ).to_csv(
            TABLE_DIR / f"final_{model_name}_{suite_name}_gate_risk.csv",
            index=False,
        )

FINAL_SUMMARY = pd.DataFrame(summary_rows)
FINAL_SUMMARY.to_csv(
    TABLE_DIR / "final_model_comparison.csv",
    index=False,
)
display(FINAL_SUMMARY)

# RF-only sweep from -15% to +15%.
RF_ERRORS = np.linspace(-0.15, 0.15, 31, dtype=np.float32)
n_rf = len(RF_ERRORS)

def zeros(*shape):
    return torch.zeros(*shape, device=DEVICE)

RF_SCENARIOS = {
    "common_B0_hz": zeros(n_rf),
    "v_offset_hz": zeros(n_rf, 3),
    "J_offset_hz": zeros(n_rf, 3),
    "rf_gain": 1.0 + torch.tensor(RF_ERRORS, device=DEVICE),
    "phase_bias_rad": zeros(n_rf),
    "clock_ppm": zeros(n_rf),
    "timing_jitter_s": zeros(n_rf, CFG.n_slices),
}

rf_rows = []
for model_name, model in models.items():
    fidelity = eval_scenarios(
        model,
        Q_FINAL[:RF_SWEEP_GATES],
        TARGET_FINAL[:RF_SWEEP_GATES],
        RF_SCENARIOS,
    )
    np.save(RAW_DIR / f"rf_sweep_{model_name}.npy", fidelity)

    for index, rf_error in enumerate(RF_ERRORS):
        rf_rows.append(
            {
                "model": model_name,
                "rf_error_percent": 100.0 * float(rf_error),
                **summarize(fidelity[:, index]),
            }
        )

RF_DF = pd.DataFrame(rf_rows)
RF_DF.to_csv(
    TABLE_DIR / "rf_sweep_nominal_vs_robust.csv",
    index=False,
)
display(RF_DF)

In [ ]:
# Publication figures.
def save_figure(name):
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / f"{name}.png", dpi=600, bbox_inches="tight")
    plt.savefig(FIGURE_DIR / f"{name}.pdf", bbox_inches="tight")
    plt.show()
    plt.close()

training_history = pd.DataFrame(history)
if len(training_history):
    plt.figure(figsize=(8, 4.8))
    plt.plot(
        training_history.global_step,
        training_history.val_nominal_mean,
        label="Nominal mean",
    )
    plt.plot(
        training_history.global_step,
        training_history.val_robust_mean,
        label="Joint robust mean",
    )
    plt.xlabel("Optimizer step")
    plt.ylabel("Fidelity")
    plt.ylim(0, 1.005)
    plt.legend()
    plt.grid(alpha=0.2)
    save_figure("robust_training_fidelity")

    plt.figure(figsize=(8, 4.8))
    plt.plot(
        training_history.global_step,
        training_history.val_cvar_infidelity_mean,
        label="Mean gate CVaR infidelity",
    )
    plt.plot(
        training_history.global_step,
        1.0 - training_history.val_gate_worst_mean,
        label="1 - mean gate-worst fidelity",
    )
    plt.xlabel("Optimizer step")
    plt.ylabel("Tail loss")
    plt.legend()
    plt.grid(alpha=0.2)
    save_figure("robust_training_tail_risk")

plt.figure(figsize=(8, 5))
for model_name in ["nominal", "robust"]:
    table = RF_DF[RF_DF.model == model_name]
    plt.plot(
        table.rf_error_percent,
        table["mean"],
        marker="o",
        ms=3,
        label=f"{model_name} mean",
    )
    plt.plot(
        table.rf_error_percent,
        table["p05"],
        ls="--",
        label=f"{model_name} p05",
    )

plt.axvline(-5, ls=":", lw=1)
plt.axvline(5, ls=":", lw=1)
plt.axvline(-10, ls=":", lw=1)
plt.axvline(10, ls=":", lw=1)
plt.xlabel("Global RF gain error (%)")
plt.ylabel("Unitary fidelity")
plt.ylim(0, 1.005)
plt.legend()
plt.grid(alpha=0.2)
save_figure("rf_sweep_nominal_vs_robust")

comparison = FINAL_SUMMARY[["model", "suite", "mean"]].pivot(
    index="suite",
    columns="model",
    values="mean",
)
comparison.plot(kind="bar", figsize=(9, 5))
plt.ylabel("Mean gate × scenario fidelity")
plt.ylim(0, 1.005)
plt.xticks(rotation=25, ha="right")
plt.grid(axis="y", alpha=0.2)
save_figure("final_suite_mean_comparison")

for suite_name in ["routine_rf05", "routine_rf10", "conservative_rf15"]:
    plt.figure(figsize=(7, 4.6))
    for model_name in ["nominal", "robust"]:
        fidelity = np.load(
            RAW_DIR / f"final_{model_name}_{suite_name}.npy"
        )
        plt.hist(
            fidelity.ravel(),
            bins=45,
            alpha=0.5,
            label=model_name,
        )

    plt.xlabel("Gate × scenario fidelity")
    plt.ylabel("Count")
    plt.title(suite_name)
    plt.legend()
    plt.grid(alpha=0.15)
    save_figure(f"hist_{suite_name}")

In [ ]:
# Export provenance, representative pulses, and checksums.
MODEL.eval()

np.save(
    RAW_DIR / "H0_full_XX_YY_ZZ.npy",
    SYS.H0.detach().cpu().numpy(),
)

with open(OUT / "model_architecture.txt", "w") as handle:
    handle.write(str(MODEL))
    handle.write("\n\n")
    handle.write(
        f"Parameters: {sum(parameter.numel() for parameter in MODEL.parameters()):,}\n"
    )

# Representative robust pulses from the final held-out target set.
final_nominal_fidelity = eval_nominal(MODEL, Q_FINAL, TARGET_FINAL)
representative_indices = {
    "worst": int(np.argmin(final_nominal_fidelity)),
    "median": int(
        np.argmin(
            np.abs(
                final_nominal_fidelity
                - np.median(final_nominal_fidelity)
            )
        )
    ),
    "best": int(np.argmax(final_nominal_fidelity)),
}

for label, index in representative_indices.items():
    with torch.no_grad():
        controls = MODEL(Q_FINAL[index : index + 1])[0].cpu().numpy()

    ux = controls[:, 0]
    uy = controls[:, 1]
    amplitude = np.sqrt(ux ** 2 + uy ** 2)
    phase = np.arctan2(uy, ux)

    np.savez(
        RAW_DIR / f"robust_pulse_{label}.npz",
        gate_index=index,
        quaternion=Q_FINAL[index].cpu().numpy(),
        fidelity=final_nominal_fidelity[index],
        ux_hz=ux,
        uy_hz=uy,
        amplitude_hz=amplitude,
        phase_rad=phase,
        dt_s=CFG.dt_s,
    )

environment = {
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "numpy": np.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
}

try:
    environment["nvidia_smi"] = subprocess.check_output(
        [
            "nvidia-smi",
            "--query-gpu=name,memory.total,driver_version",
            "--format=csv",
        ],
        text=True,
    )
except Exception as exc:
    environment["nvidia_smi_error"] = str(exc)

with open(OUT / "environment.json", "w") as handle:
    json.dump(environment, handle, indent=2)

report = {
    "source_nominal_checkpoint_sha256": sha256_file(NOMINAL_CHECKPOINT),
    "source_nominal_validation": nominal_checkpoint.get("val_stats"),
    "selected_robust_checkpoint": best_record,
    "final_comparison": FINAL_SUMMARY.to_dict(orient="records"),
    "training_rf_abs_fraction": TRAIN_RF_ABS,
    "note": (
        "Training used the ±5% RF envelope only. "
        "The ±7.5%, ±10%, and conservative ±15% envelopes are evaluation-only."
    ),
}

with open(OUT / "final_report.json", "w") as handle:
    json.dump(report, handle, indent=2, default=str)

with open(OUT / "README_RESULTS.txt", "w") as handle:
    handle.write("RISK-AWARE ROBUST FULL-HAAR NMR COMPILER\n")
    handle.write("=" * 48 + "\n\n")
    handle.write("Training RF envelope: ±5%\n")
    handle.write(f"Selected optimizer step: {best_record['global_step']}\n\n")
    handle.write(FINAL_SUMMARY.to_string(index=False))
    handle.write(
        "\n\nThe wider RF envelopes are evaluation-only tests.\n"
    )

manifest = [
    str(path.relative_to(OUT))
    for path in sorted(OUT.rglob("*"))
    if path.is_file()
]
with open(OUT / "MANIFEST.txt", "w") as handle:
    handle.write("\n".join(manifest))

checksum_lines = []
for path in sorted(OUT.rglob("*")):
    if path.is_file() and path.name != "SHA256SUMS.txt":
        checksum_lines.append(
            f"{sha256_file(path)}  {path.relative_to(OUT)}"
        )

with open(OUT / "SHA256SUMS.txt", "w") as handle:
    handle.write("\n".join(checksum_lines))

print("Robust-training outputs:", OUT)
print("Files written:", sum(1 for path in OUT.rglob("*") if path.is_file()))